# 04 - Review Sentiment Analysis

In this notebook, we use **Natural Language Processing (NLP)** to understand what players really think about their games. Using TextBlob, we score the sentiment of Steam user reviews and explore how those sentiment scores relate to game success metrics like recommendations and Metacritic scores.

**Key questions we'll answer:**
- How is review sentiment distributed across Steam?
- Does higher sentiment correlate with more recommendations?
- Which genres have the happiest (and unhappiest) players?
- How well does TextBlob polarity align with the thumbs-up/thumbs-down vote?
- Is review sentiment trending up or down over the years?

In [ ]:
# -- Setup: imports and styling --
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.data_loader import (load_applications, load_reviews, build_games_with_genres,
                              STEAM_COLORS, CHART_COLORS)
from src.utils import setup_matplotlib_style, get_plotly_layout

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from textblob import TextBlob
import warnings
warnings.filterwarnings('ignore')

# Apply our Steam dark theme to matplotlib
setup_matplotlib_style()

print("All imports loaded successfully!")

In [ ]:
# -- Load data --
# We sample 100k reviews to keep things fast (full dataset is millions of rows)
reviews = load_reviews(sample_size=100000)
games = load_applications()

print(f"\nReviews loaded: {len(reviews):,} rows")
print(f"Games loaded:   {len(games):,} rows")
print(f"\nReview columns: {list(reviews.columns)}")

## 1. Sentiment Scoring

We'll use **TextBlob** to calculate two sentiment metrics for each review:
- **Polarity** (-1 to 1): How positive or negative the text is
- **Subjectivity** (0 to 1): How opinionated vs factual the text is

In [ ]:
# -- Apply TextBlob sentiment to English reviews --

# Filter to English reviews only (TextBlob works best with English)
english_reviews = reviews[reviews['language'] == 'english'].copy()
print(f"English reviews: {len(english_reviews):,} out of {len(reviews):,} total")
print(f"That's {len(english_reviews)/len(reviews)*100:.1f}% of our sample\n")

# Simple function to get sentiment scores from text
def get_sentiment(text):
    """Return (polarity, subjectivity) for a piece of text."""
    try:
        blob = TextBlob(str(text))
        return blob.sentiment.polarity, blob.sentiment.subjectivity
    except:
        return 0.0, 0.0

# Apply sentiment scoring to each review
print("Calculating sentiment scores (this may take a minute)...")
sentiment_results = english_reviews['review_text'].apply(get_sentiment)

# Unpack the (polarity, subjectivity) tuples into separate columns
english_reviews['polarity'] = sentiment_results.apply(lambda x: x[0])
english_reviews['subjectivity'] = sentiment_results.apply(lambda x: x[1])

print(f"Done! Scored {len(english_reviews):,} reviews.\n")

# Show a sample of reviews with their sentiment scores
print("Sample reviews with sentiment scores:")
sample_cols = ['review_text', 'polarity', 'subjectivity', 'voted_up']
print(english_reviews[sample_cols].head(5).to_string(max_colwidth=80))

# Quick stats
print(f"\nAverage polarity:     {english_reviews['polarity'].mean():.3f}")
print(f"Average subjectivity: {english_reviews['subjectivity'].mean():.3f}")

In [ ]:
# -- Distribution of sentiment polarity --

fig = go.Figure()

# Histogram of polarity scores
fig.add_trace(go.Histogram(
    x=english_reviews['polarity'],
    nbinsx=50,
    marker_color=STEAM_COLORS['light_blue'],
    opacity=0.8,
    name='Review Polarity'
))

# Add vertical lines to show negative / neutral / positive zones
# Negative zone boundary
fig.add_vline(x=-0.05, line_dash='dash', line_color=STEAM_COLORS['accent_red'],
              annotation_text='Negative', annotation_position='top left',
              annotation_font_color=STEAM_COLORS['accent_red'])
# Positive zone boundary
fig.add_vline(x=0.05, line_dash='dash', line_color=STEAM_COLORS['accent_green'],
              annotation_text='Positive', annotation_position='top right',
              annotation_font_color=STEAM_COLORS['accent_green'])
# Zero line (neutral)
fig.add_vline(x=0, line_dash='solid', line_color=STEAM_COLORS['accent_orange'],
              annotation_text='Neutral', annotation_position='bottom',
              annotation_font_color=STEAM_COLORS['accent_orange'])

fig.update_layout(get_plotly_layout(
    'Distribution of Review Sentiment Polarity',
    xaxis_title='Polarity (-1 = Negative, +1 = Positive)',
    yaxis_title='Number of Reviews'
))

fig.show()

# Print the breakdown
negative = (english_reviews['polarity'] < -0.05).sum()
neutral = ((english_reviews['polarity'] >= -0.05) & (english_reviews['polarity'] <= 0.05)).sum()
positive = (english_reviews['polarity'] > 0.05).sum()
total = len(english_reviews)

print(f"Negative (< -0.05): {negative:,} reviews ({negative/total*100:.1f}%)")
print(f"Neutral  (-0.05 to 0.05): {neutral:,} reviews ({neutral/total*100:.1f}%)")
print(f"Positive (> 0.05):  {positive:,} reviews ({positive/total*100:.1f}%)")

## 2. Sentiment vs Game Metrics

Let's see if games with more positive reviews also tend to have more recommendations and higher Metacritic scores.

In [ ]:
# -- Merge reviews with game data and compute per-game averages --

# Merge reviews with games to get game names and metrics
reviews_with_games = english_reviews.merge(
    games[['appid', 'name', 'recommendations_total', 'metacritic_score']],
    on='appid',
    how='inner'
)
print(f"Matched {len(reviews_with_games):,} reviews to game info\n")

# Group by game: calculate average sentiment, review count, and game metrics
game_sentiment = reviews_with_games.groupby('appid').agg(
    avg_sentiment=('polarity', 'mean'),
    review_count=('polarity', 'count'),
    game_name=('name', 'first'),
    recommendations_total=('recommendations_total', 'first'),
    metacritic_score=('metacritic_score', 'first')
).reset_index()

# Only look at games with enough reviews for a reliable average
game_sentiment = game_sentiment[game_sentiment['review_count'] >= 5].copy()
print(f"Games with 5+ reviews: {len(game_sentiment):,}\n")

# Top 10 most positively reviewed games
top_positive = game_sentiment.nlargest(10, 'avg_sentiment')
print("Top 10 Most Positively Reviewed Games:")
print("=" * 70)
for i, row in top_positive.iterrows():
    print(f"  {row['game_name'][:40]:<42} sentiment: {row['avg_sentiment']:.3f}  "
          f"({row['review_count']} reviews)")

In [ ]:
# -- Scatter plot: Average sentiment vs total recommendations --

# Filter out games with zero recommendations for the log scale
scatter_data = game_sentiment[game_sentiment['recommendations_total'] > 0].copy()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=scatter_data['avg_sentiment'],
    y=scatter_data['recommendations_total'],
    mode='markers',
    marker=dict(
        size=6,
        color=scatter_data['avg_sentiment'],
        colorscale='RdYlGn',   # Red for negative, green for positive
        colorbar=dict(title='Avg Sentiment'),
        opacity=0.6,
        line=dict(width=0.5, color='white')
    ),
    text=scatter_data['game_name'],
    hovertemplate='<b>%{text}</b><br>Sentiment: %{x:.3f}<br>Recommendations: %{y:,}<extra></extra>'
))

layout = get_plotly_layout(
    'Average Sentiment vs Total Recommendations',
    xaxis_title='Average Sentiment Polarity',
    yaxis_title='Total Recommendations (log scale)'
)
layout.update(yaxis=dict(
    type='log',
    gridcolor='rgba(198, 213, 224, 0.15)',
    zerolinecolor='rgba(198, 213, 224, 0.3)',
    title='Total Recommendations (log scale)'
))

fig.update_layout(layout)
fig.show()

# Calculate correlation
corr = scatter_data['avg_sentiment'].corr(scatter_data['recommendations_total'])
print(f"Correlation between avg sentiment and recommendations: {corr:.3f}")

## 3. Sentiment by Genre

Do certain genres tend to attract more positive or negative reviews? Let's find out which types of games make players the happiest.

In [ ]:
# -- Box plot: sentiment distribution by genre --

# Build the games-with-genres dataset
games_genres = build_games_with_genres(games)

# Merge reviews with genre information
reviews_genres = english_reviews.merge(
    games_genres[['appid', 'genre_name']],
    on='appid',
    how='inner'
)
print(f"Reviews with genre info: {len(reviews_genres):,}\n")

# Find the top 10 genres by review count
top_genres = reviews_genres['genre_name'].value_counts().head(10).index.tolist()
print(f"Top 10 genres by review count: {top_genres}\n")

# Filter to only top genres
genre_data = reviews_genres[reviews_genres['genre_name'].isin(top_genres)].copy()

# Create box plot
fig = go.Figure()

for i, genre in enumerate(top_genres):
    genre_reviews = genre_data[genre_data['genre_name'] == genre]['polarity']
    fig.add_trace(go.Box(
        y=genre_reviews,
        name=genre,
        marker_color=CHART_COLORS[i % len(CHART_COLORS)],
        boxmean=True  # Show the mean as a dashed line
    ))

fig.update_layout(get_plotly_layout(
    'Sentiment Distribution by Genre (Top 10)',
    xaxis_title='Genre',
    yaxis_title='Sentiment Polarity'
))
fig.update_layout(showlegend=False)

fig.show()

In [ ]:
# -- Bar chart: average sentiment per genre, ranked --

# Calculate average sentiment for each genre
genre_avg = reviews_genres.groupby('genre_name')['polarity'].agg(['mean', 'count']).reset_index()
genre_avg.columns = ['genre_name', 'avg_sentiment', 'review_count']

# Only include genres with at least 100 reviews for reliability
genre_avg = genre_avg[genre_avg['review_count'] >= 100].copy()

# Sort by average sentiment
genre_avg = genre_avg.sort_values('avg_sentiment', ascending=True)

# Color bars: green for positive, red for negative
bar_colors = [STEAM_COLORS['accent_green'] if x > 0 else STEAM_COLORS['accent_red']
              for x in genre_avg['avg_sentiment']]

fig = go.Figure()

fig.add_trace(go.Bar(
    y=genre_avg['genre_name'],
    x=genre_avg['avg_sentiment'],
    orientation='h',
    marker_color=bar_colors,
    text=genre_avg['avg_sentiment'].apply(lambda x: f'{x:.3f}'),
    textposition='outside',
    textfont=dict(color='#c7d5e0'),
    hovertemplate='<b>%{y}</b><br>Avg Sentiment: %{x:.3f}<extra></extra>'
))

fig.update_layout(get_plotly_layout(
    'Average Sentiment by Genre (Ranked)',
    xaxis_title='Average Sentiment Polarity',
    yaxis_title=''
))
fig.update_layout(height=600)

fig.show()

# Print the happiest and unhappiest genres
print("Happiest genre:    ", genre_avg.iloc[-1]['genre_name'],
      f"({genre_avg.iloc[-1]['avg_sentiment']:.3f})")
print("Unhappiest genre:  ", genre_avg.iloc[0]['genre_name'],
      f"({genre_avg.iloc[0]['avg_sentiment']:.3f})")

## 4. Thumbs Up vs Sentiment Alignment

Steam reviews have a **thumbs up / thumbs down** vote. Let's see how well TextBlob's sentiment polarity agrees with the player's actual vote.

In [ ]:
# -- Compare voted_up (True/False) with TextBlob sentiment --

# Classify each review's sentiment as positive or negative
english_reviews['sentiment_positive'] = english_reviews['polarity'] > 0

# Check agreement: does the sentiment match the vote?
english_reviews['agrees'] = english_reviews['sentiment_positive'] == english_reviews['voted_up']

agreement_rate = english_reviews['agrees'].mean() * 100
print(f"TextBlob agrees with thumbs up/down: {agreement_rate:.1f}% of the time\n")

# Break it down further
thumbs_up = english_reviews[english_reviews['voted_up'] == True]
thumbs_down = english_reviews[english_reviews['voted_up'] == False]

print(f"Thumbs UP reviews:   avg sentiment = {thumbs_up['polarity'].mean():.3f}  "
      f"(n={len(thumbs_up):,})")
print(f"Thumbs DOWN reviews: avg sentiment = {thumbs_down['polarity'].mean():.3f}  "
      f"(n={len(thumbs_down):,})")

# Grouped bar chart: avg sentiment for thumbs up vs thumbs down
vote_sentiment = english_reviews.groupby('voted_up')['polarity'].mean().reset_index()
vote_sentiment['vote_label'] = vote_sentiment['voted_up'].map(
    {True: 'Thumbs Up', False: 'Thumbs Down'}
)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=vote_sentiment['vote_label'],
    y=vote_sentiment['polarity'],
    marker_color=[STEAM_COLORS['accent_red'], STEAM_COLORS['accent_green']],
    text=vote_sentiment['polarity'].apply(lambda x: f'{x:.3f}'),
    textposition='outside',
    textfont=dict(color='#c7d5e0', size=14),
    hovertemplate='<b>%{x}</b><br>Avg Sentiment: %{y:.3f}<extra></extra>'
))

fig.update_layout(get_plotly_layout(
    'Average Sentiment: Thumbs Up vs Thumbs Down Reviews',
    xaxis_title='Vote Type',
    yaxis_title='Average Polarity'
))

fig.show()

print(f"\nThe difference in avg sentiment between thumbs up and down: "
      f"{thumbs_up['polarity'].mean() - thumbs_down['polarity'].mean():.3f}")

## 5. Sentiment Over Time

Are Steam reviews getting more positive or negative over the years? Let's track average sentiment over time.

In [ ]:
# -- Sentiment over time: group by review year --

# Extract the year from the review date
english_reviews['review_year'] = english_reviews['review_date'].dt.year

# Group by year and calculate average sentiment
yearly_sentiment = english_reviews.groupby('review_year').agg(
    avg_sentiment=('polarity', 'mean'),
    review_count=('polarity', 'count')
).reset_index()

# Drop years with very few reviews (likely bad data) and future years
yearly_sentiment = yearly_sentiment[
    (yearly_sentiment['review_count'] >= 50) &
    (yearly_sentiment['review_year'] <= 2025)
].copy()

print("Average Sentiment by Year:")
print("=" * 45)
for _, row in yearly_sentiment.iterrows():
    print(f"  {int(row['review_year'])}:  sentiment = {row['avg_sentiment']:.3f}  "
          f"({row['review_count']:,} reviews)")

# Line chart
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=yearly_sentiment['review_year'],
    y=yearly_sentiment['avg_sentiment'],
    mode='lines+markers',
    line=dict(color=STEAM_COLORS['light_blue'], width=3),
    marker=dict(size=8, color=STEAM_COLORS['light_blue'],
                line=dict(width=1, color='white')),
    hovertemplate='Year: %{x}<br>Avg Sentiment: %{y:.3f}<extra></extra>'
))

fig.update_layout(get_plotly_layout(
    'Average Review Sentiment Over Time',
    xaxis_title='Year',
    yaxis_title='Average Polarity'
))

# Make the x-axis show integer years
fig.update_xaxes(dtick=1)

fig.show()

## Summary

### Key Insights

1. **Genre happiness:** Some genres consistently receive more positive reviews than others. This could help developers understand player expectations when choosing what type of game to build.

2. **Sentiment and success:** The relationship between average sentiment and total recommendations reveals whether happier players lead to more popular games, or if highly popular games attract a wider (and sometimes more critical) audience.

3. **Thumbs up alignment:** TextBlob's sentiment polarity and the player's actual thumbs-up/down vote don't always agree. This makes sense -- players often write sarcastic, ironic, or nuanced reviews that simple polarity scoring can miss.

4. **Trends over time:** Tracking sentiment year-over-year shows whether the Steam community is becoming more or less positive in its reviews.

### Limitations of TextBlob

- **Sarcasm and irony:** TextBlob can't detect sarcasm (e.g., "Great, another bug" would score as positive).
- **Gaming jargon:** Words like "sick," "insane," or "broken" have different meanings in gaming contexts that TextBlob doesn't understand.
- **Short reviews:** Many Steam reviews are very short ("good game" or just "no"), giving extreme polarity scores.
- **Non-English text:** We filtered to English only, which means we're missing a large portion of the global Steam community.

For a production system, a fine-tuned transformer model (like BERT trained on game reviews) would give much better results. But TextBlob is a great starting point for understanding the general landscape of player sentiment!